# Lab 09: Multi-Agent Systems and Human-AI Cooperation

**Course:** ISDN 3150 / ISDN 5005C — AI for Design, 2026 Spring  
**API:** HKUST Azure OpenAI (`gpt-5-mini`)  

---

***>>> What you will learn in this lab:***
- How to design agentic workflows by mapping human actions to LLM actions
- Three levels of reflection: no reflection, self-reflection, and external reflection
- How to build a planning agent that constructs and executes its own plan
- How to build multi-agent systems using AutoGen (manager pattern & group chat)

---

| Section | Topic |
|---------|-------|
| 0 | Setup & Environment |
| 1 | Agentic Workflow Design: Human Action → LLM Action |
| 2 | Reflection: Non-Reflection / Self-Reflection / External Reflection |
| 3 | Planning: Construct a Plan and Execute It |
| 4 | Multi-Agent: Manager Pattern & Group Chat (AutoGen) |
| — | Wrap-Up & Key Takeaways |

---
## Section 0: Setup & Environment

### 0.1 Install Dependencies

We need two packages:
- `openai` — for calling the HKUST Azure OpenAI API directly
- `autogen-agentchat` + `autogen-ext[openai]` — for building multi-agent systems in Section 4

In [ ]:
!pip install -q openai autogen-agentchat autogen-ext[openai]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.3/119.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.9/101.9 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 16.4 MB/s eta 0:00:00


### 0.2 Connect to HKUST Azure OpenAI API

We use the HKUST GenAI API endpoint. This is an **Azure OpenAI** deployment, so we use `AzureOpenAI` instead of the regular `OpenAI` client.

We also define a `chat()` helper function that we'll reuse throughout the lab. This wraps the API call so we don't have to repeat the boilerplate every time.

In [ ]:
from openai import AzureOpenAI

# ── HKUST Azure OpenAI Configuration ──────────────────────────
# These credentials connect to HKUST's Azure OpenAI deployment.
# The model 'gpt-5-mini' is a fast, cost-efficient model suitable
# for agentic workflows where we make many API calls.
AZURE_API_KEY = ""
AZURE_ENDPOINT = "https://hkust.azure-api.net"
AZURE_API_VERSION = "2025-02-01-preview"
MODEL_NAME = "gpt-5-mini"

# Create the Azure OpenAI client
client = AzureOpenAI(
    api_key=AZURE_API_KEY,
    api_version=AZURE_API_VERSION,
    azure_endpoint=AZURE_ENDPOINT,
)

def chat(messages, tools=None, tool_choice=None):
    """
    Wrapper for Azure OpenAI chat completion.

    Args:
        messages: List of message dicts [{"role": ..., "content": ...}]
        tools: Optional list of tool/function definitions for function calling
        tool_choice: Optional tool selection strategy

    Returns:
        The assistant's response message object.
    """
    kwargs = dict(
        model=MODEL_NAME,
        messages=messages,
    )
    if tools:
        kwargs["tools"] = tools
        if tool_choice:
            kwargs["tool_choice"] = tool_choice
    response = client.chat.completions.create(**kwargs)
    return response.choices[0].message

# ── Quick connectivity test ───────────────────────────────────
reply = chat([{"role": "user", "content": "Say 'Lab 09 is ready!' in one sentence."}])
print(reply.content)
print("\n✓ API connection successful!")

Lab 09 is ready!

✓ API connection successful!


---
## Section 1: Agentic Workflow Design — Human Action → LLM Action

### 1.1 Key Idea

The core design recipe for any agentic workflow:

1. **Summarize the human workflow** — How would a person do this task, step by step?
2. **Map each step to an LLM action** — Which steps can a prompt handle?
3. **Decompose further** — If a step's output isn't good enough, break it into smaller sub-steps.

An agentic workflow follows this loop:

```
Perception → Reasoning → Action → Observation/Feedback
     ↑                                        |
     └────────────────────────────────────────┘
```

- **Perception:** The agent receives input (user's request, a file, a webpage)
- **Reasoning:** It breaks the goal into sub-tasks (Chain of Thought)
- **Action:** It uses tools or generates output
- **Observation/Feedback:** It evaluates results; loops back if not done

### 1.2 Example: Writing a Product Description

| # | Human Action | → | LLM Action |
|---|-------------|---|------------|
| 1 | Research product features | → | Provide product info in prompt (Perception) |
| 2 | Write a first draft | → | LLM generates draft (Reasoning + Action) |
| 3 | Read it back, find problems | → | LLM critiques the draft (Observation) |
| 4 | Revise based on feedback | → | LLM revises (loop back) |

Let's see both approaches in action — **non-agentic** (single pass) vs **agentic** (multi-step with feedback).

> **What to observe:** Run both cells below and compare the outputs. The agentic version should be noticeably more polished because it self-corrects.

In [ ]:
# ══════════════════════════════════════════════════════════════
# APPROACH A: Non-Agentic (Single Pass)
# This is like asking a human to write something once with no
# chance to revise. You get whatever comes out on the first try.
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("APPROACH A: Non-Agentic (Single Pass)")
print("=" * 60)

response_a = chat([
    {"role": "user", "content": (
        "Write a 50-word product description for a portable "
        "Bluetooth speaker called 'SoundWave Mini'."
    )}
])
print(response_a.content)

APPROACH A: Non-Agentic (Single Pass)
SoundWave Mini delivers rich, room-filling audio from a compact, travel-ready design. Bluetooth pairing is instant and stable, offering eight hours of playback on a single charge. Rugged, water-resistant housing and hands-free calling make it perfect for outdoor adventures, commutes, and small gatherings—big sound, tiny package. Ready for every soundtrack anytime.


In [ ]:
# ══════════════════════════════════════════════════════════════
# APPROACH B: Agentic Workflow (Draft → Critique → Revise)
#
# This maps the human writing process to LLM actions:
#   Human writes a draft  →  LLM generates text
#   Human re-reads it     →  LLM critiques its own output
#   Human revises it      →  LLM revises based on the critique
#
# Key insight: Same LLM, but the multi-step process produces
# better results because each step has a focused objective.
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("APPROACH B: Agentic Workflow (Draft → Critique → Revise)")
print("=" * 60)

# ── Step 1: Draft ─────────────────────────────────────────────
# Human action: "Write a first draft"
# LLM action:   Generate text from the prompt
draft = chat([
    {"role": "user", "content": (
        "Write a 50-word product description for a portable "
        "Bluetooth speaker called 'SoundWave Mini'."
    )}
])
print(f"[Step 1 — Draft]\n{draft.content}\n")

# ── Step 2: Critique ──────────────────────────────────────────
# Human action: "Read it back, spot problems"
# LLM action:   Evaluate the draft against specific criteria
critique = chat([
    {"role": "user", "content": (
        f"Critique the following product description. "
        f"Is it persuasive? Is it exactly 50 words? "
        f"List specific improvements.\n\n"
        f"Description:\n{draft.content}"
    )}
])
print(f"[Step 2 — Critique]\n{critique.content}\n")

# ── Step 3: Revise ────────────────────────────────────────────
# Human action: "Fix the problems found in step 2"
# LLM action:   Rewrite incorporating the critique feedback
revised = chat([
    {"role": "user", "content": (
        f"Revise the following product description based on the critique below. "
        f"Make it exactly 50 words, persuasive, and polished.\n\n"
        f"Original:\n{draft.content}\n\n"
        f"Critique:\n{critique.content}"
    )}
])
print(f"[Step 3 — Revised]\n{revised.content}")

APPROACH B: Agentic Workflow (Draft → Critique → Revise)
[Step 1 — Draft]
SoundWave Mini is a compact portable Bluetooth speaker delivering rich room filling sound with deep bass. Rugged splashproof design, twelve hour battery life, and seamless wireless pairing make it ideal for travel, outdoor adventures, and home. Intuitive controls and built in microphone enable hands free calling and voice assistant access.

[Step 2 — Critique]
Yes — the description is exactly 50 words.

Persuasiveness: moderate. Strengths: concise, covers key features (portability, battery, pairing, microphone). Weaknesses: vague claims, mechanical wording, inconsistent punctuation, and missed opportunities to highlight unique benefits or proof points.

Specific improvements
- Hyphenate compound adjectives: "room-filling," "twelve-hour," "built-in," "hands-free," "voice-assistant."
- Use numerals for clarity: "12-hour battery" (or state mAh) rather than spelled-out words.
- Add measurable specs or standards to bu

### 1.3 Decomposition: Breaking Down Complex Tasks

For example, "Write a blog post" can be decomposed into finer human actions:

| # | Human Action | → | LLM Action |
|---|-------------|---|------------|
| 1 | Research the topic | → | LLM brainstorms key points |
| 2 | Create an outline | → | LLM generates structured outline |
| 3 | Write the full post | → | LLM writes from the outline |
| 4 | Proofread and polish | → | LLM checks grammar and word count |

Each step has a **narrow, focused objective** — this is why decomposition works.

In [ ]:
# ══════════════════════════════════════════════════════════════
# DECOMPOSED AGENTIC WORKFLOW: Blog Post Writing
#
# We decompose "write a blog post" into 4 focused steps.
# Each step's output feeds into the next step as context.
# This mirrors how a professional writer actually works.
# ══════════════════════════════════════════════════════════════
TOPIC = "Why AI Agents Are the Future of Software Design"

# ── Step 1: Brainstorm ────────────────────────────────────────
# Human: "What are the main points I want to make?"
print("[Step 1: Brainstorm Key Points]")
brainstorm = chat([{
    "role": "user",
    "content": f"Brainstorm 5 key points for a short blog post about: '{TOPIC}'. Return a numbered list."
}])
print(brainstorm.content)
print()

# ── Step 2: Outline ───────────────────────────────────────────
# Human: "Let me organize these points into a structure."
# Note: We pass the brainstorm output as context.
print("[Step 2: Create Outline]")
outline = chat([{
    "role": "user",
    "content": (
        f"Create a structured outline for a 300-word blog post based on these key points:\n\n"
        f"{brainstorm.content}\n\n"
        f"Include: title, introduction, 3 body sections, conclusion."
    )
}])
print(outline.content)
print()

# ── Step 3: Write ─────────────────────────────────────────────
# Human: "Now I'll flesh out each section of the outline."
# Note: We pass the outline as context — the LLM follows it.
print("[Step 3: Write Full Post]")
full_post = chat([{
    "role": "user",
    "content": (
        f"Write a 300-word blog post following this outline exactly:\n\n"
        f"{outline.content}\n\n"
        f"Make it engaging, use a professional but friendly tone."
    )
}])
print(full_post.content)
print()

# ── Step 4: Proofread ─────────────────────────────────────────
# Human: "Let me re-read and fix any issues before publishing."
print("[Step 4: Proofread & Finalize]")
final = chat([{
    "role": "user",
    "content": (
        f"Proofread the following blog post. Fix any grammar issues, "
        f"improve clarity, and ensure it's exactly 300 words (±20 words).\n\n"
        f"{full_post.content}"
    )
}])
print(final.content)

[Step 1: Brainstorm Key Points]
1. Autonomous, end-to-end problem solving — AI agents can interpret requirements, plan multi-step workflows, write and test code, and deploy or integrate features, reducing manual orchestration and speeding delivery.

2. Composability and orchestration — agents act as modular building blocks that can be chained, delegated, or orchestrated to assemble complex systems rapidly, making architectures more flexible and reusable.

3. Continuous learning and self-improvement — agents can monitor behavior, run diagnostics, fix regressions, and optimize performance over time, enabling software that adapts automatically as usage and requirements change.

4. Democratization of development — natural-language interfaces and automation of repetitive boilerplate let domain experts and non‑developers contribute directly, lowering the barrier to building and iterating on software.

5. Faster innovation and cost efficiency — by offloading routine tasks and accelerating exp

### 1.4 Discussion

**Questions to think about:**

1. **Quality:** Compare Approach A (single pass) vs. Approach B (agentic loop). Which produced a better product description? Why?
   - *Expected answer:* The agentic version is better because the critique step identifies specific weaknesses, and the revise step addresses them.

2. **Decomposition:** How did breaking "write a blog post" into 4 steps improve the result?
   - *Expected answer:* Each step has a narrow focus (brainstorm ≠ outline ≠ write ≠ proofread), so the LLM can give its full attention to one task at a time.

3. **Trade-offs:** What's the cost of an agentic approach?
   - *Expected answer:* More API calls = more cost + higher latency. The agentic version used 3 calls vs. 1. The decomposed blog post used 4 calls. You need to balance quality vs. cost.

**Key takeaway:** The design principle is always the same — *think about how a human would do it, map to LLM actions, decompose if needed.*

---
## Section 2: Reflection — Non-Reflection / Self-Reflection / External Reflection

Reflection mimics how humans iterate on their work: **draft → review → revise**.

We'll compare three levels, from weakest to strongest:

| Level | Method | Feedback Source | Reliability |
|-------|--------|----------------|-------------|
| **Non-Reflection** | Single pass, no review | None | Low |
| **Self-Reflection** | LLM critiques its own output | The LLM itself | Medium |
| **External Reflection** | External tool verifies output | Real-world execution | **Highest** |

We'll use the same task for all three: **write a palindrome checker function in Python.**

### 2.1 Level 1: Non-Reflection (Baseline)

Just ask the LLM once, take whatever it gives us. No review, no iteration.

In [ ]:
# ══════════════════════════════════════════════════════════════
# LEVEL 1: NON-REFLECTION — Single pass, no review
#
# This is the baseline. We ask the LLM once and accept the
# result without any verification. It might be correct, or
# it might have subtle bugs — we have no way to know.
# ══════════════════════════════════════════════════════════════

# We define the task once and reuse it across all 3 levels
CODING_TASK = (
    "Write a Python function `is_palindrome(s: str) -> bool` that checks "
    "if a string is a palindrome. Ignore spaces, punctuation, and case."
)

print("=" * 60)
print("Level 1: NON-REFLECTION (Single Pass)")
print("=" * 60)

response = chat([
    {"role": "user", "content": f"{CODING_TASK}\n\nReturn ONLY the Python function."}
])

print(response.content)
print("\n→ No review step. We just trust the first output.")
print("→ Is it correct? We don't know! We'd have to test it manually.")

Level 1: NON-REFLECTION (Single Pass)
def is_palindrome(s: str) -> bool:
    cleaned = ''.join(ch.lower() for ch in s if ch.isalnum())
    return cleaned == cleaned[::-1]

→ No review step. We just trust the first output.
→ Is it correct? We don't know! We'd have to test it manually.


### 2.2 Level 2: Self-Reflection

> **What to observe:** Watch how the LLM finds issues in its own code that it didn't catch on the first pass. This is powerful — but the LLM can also *miss* real bugs or *hallucinate* false issues.

The LLM generates code, then **critiques its own output** and iterates. The loop continues until the self-critique says "APPROVED" or we hit a maximum number of iterations.

In [ ]:
# ══════════════════════════════════════════════════════════════
# LEVEL 2: SELF-REFLECTION — LLM critiques its own output
#
# Pattern:
#   1. LLM generates code
#   2. LLM reviews its own code (using a critique prompt)
#   3. If issues found → LLM revises and loops back to step 2
#   4. If "APPROVED" → stop
#
# This is like a developer doing code review on their own work.
# Better than nothing, but they might miss their own blind spots.
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("Level 2: SELF-REFLECTION")
print("=" * 60)

MAX_ITERATIONS = 3  # Safety limit to prevent infinite loops
messages = [{"role": "user", "content": CODING_TASK}]

for i in range(MAX_ITERATIONS):
    print(f"\n{'─'*40}")
    print(f"Iteration {i + 1}")
    print(f"{'─'*40}")

    # ── Generate / Revise ──────────────────────────────────────
    response = chat(messages)
    print(f"\n[Generation]\n{response.content}")
    messages.append({"role": "assistant", "content": response.content})

    # ── Self-Critique ─────────────────────────────────────────
    # We explicitly ask the LLM to check specific quality criteria.
    # The "APPROVED" keyword gives us a clear termination signal.
    critique_prompt = (
        "Review the code you just wrote. Check for:\n"
        "1. Correctness: Does it handle edge cases (empty string, single char, punctuation)?\n"
        "2. Efficiency: Is the approach optimal?\n"
        "3. Code quality: Are variable names clear?\n\n"
        "If the code is perfect, respond with exactly: 'APPROVED'.\n"
        "Otherwise, list the issues and provide a corrected version."
    )
    messages.append({"role": "user", "content": critique_prompt})

    critique = chat(messages)
    print(f"\n[Self-Critique]\n{critique.content}")
    messages.append({"role": "assistant", "content": critique.content})

    # ── Check termination ─────────────────────────────────────
    if "APPROVED" in critique.content.upper():
        print("\n✓ Code approved after self-reflection!")
        break
    else:
        # Ask for a revised version incorporating the critique
        messages.append({
            "role": "user",
            "content": "Please provide the corrected version based on your critique above."
        })

print("\n--- Self-Reflection Complete ---")

Level 2: SELF-REFLECTION

────────────────────────────────────────
Iteration 1
────────────────────────────────────────

[Generation]
Here's a simple, clear implementation that ignores spaces, punctuation, and case:

```python
def is_palindrome(s: str) -> bool:
    """
    Return True if s is a palindrome, ignoring spaces, punctuation, and case.
    """
    cleaned = ''.join(ch.lower() for ch in s if ch.isalnum())
    return cleaned == cleaned[::-1]
```

Examples:
- is_palindrome("A man, a plan, a canal: Panama") -> True
- is_palindrome("No 'x' in Nixon") -> True
- is_palindrome("hello") -> False

Notes:
- .isalnum() keeps letters and digits and ignores punctuation and whitespace.
- This approach builds a cleaned string; for very long inputs you can use a two-pointer approach to avoid extra memory.

[Self-Critique]
Issues:
- Use of str.lower() rather than str.casefold() can fail for some Unicode case-mapping edge cases. Use casefold() for more correct Unicode-insensitive comparisons.
-

### 2.3 Level 3: External Reflection

> **What to observe:** This time, feedback doesn't come from the LLM — it comes from **actually running the code** with test cases. If a test fails, the real Python error message is fed back to the LLM. This is the most reliable form of reflection.

Instead of trusting the LLM's self-assessment, we run the code against real test cases. The Python interpreter provides **ground truth** that the LLM cannot hallucinate away.

In [ ]:
import subprocess
import textwrap
import tempfile
import os

# ══════════════════════════════════════════════════════════════
# LEVEL 3: EXTERNAL REFLECTION — Real test execution
#
# Pattern:
#   1. LLM generates code
#   2. We ACTUALLY RUN the code with test cases
#   3. If tests pass → done!
#   4. If tests fail → feed the REAL error message back to LLM
#   5. LLM fixes the bug based on real feedback → loop
#
# This is like a developer with a CI/CD pipeline: the code
# must pass automated tests before it's accepted.
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("Level 3: EXTERNAL REFLECTION (Test Execution)")
print("=" * 60)

# These test cases serve as our "external oracle" — they define
# what correct behavior looks like, independent of the LLM.
TEST_CODE = textwrap.dedent("""
assert is_palindrome("racecar") == True
assert is_palindrome("A man a plan a canal Panama") == True
assert is_palindrome("hello") == False
assert is_palindrome("Was it a car or a cat I saw?") == True
assert is_palindrome("") == True
assert is_palindrome("a") == True
print("ALL TESTS PASSED")
""")

MAX_ATTEMPTS = 3
messages = [
    {"role": "user", "content": (
        f"{CODING_TASK}\n\nReturn ONLY the Python function, no explanation."
    )}
]

for attempt in range(MAX_ATTEMPTS):
    print(f"\n{'─'*40}")
    print(f"Attempt {attempt + 1}")
    print(f"{'─'*40}")

    response = chat(messages)
    code = response.content

    # Extract code from markdown block if the LLM wraps it
    if "```python" in code:
        code = code.split("```python")[1].split("```")[0]
    elif "```" in code:
        code = code.split("```")[1].split("```")[0]

    print(f"\n[Generated Code]\n{code}")

    # ── EXTERNAL VERIFICATION ─────────────────────────────────
    # We write the code + tests to a temp file and run it.
    # The subprocess gives us real stdout/stderr from Python.
    full_code = code + "\n" + TEST_CODE
    with tempfile.NamedTemporaryFile(mode='w', suffix='.py', delete=False) as f:
        f.write(full_code)
        tmp_path = f.name

    try:
        result = subprocess.run(
            ["python3", tmp_path],
            capture_output=True, text=True, timeout=5
        )
        if result.returncode == 0:
            print(f"\n[External Test Output]\n{result.stdout}")
            print("✓ All tests passed — code verified by REAL execution!")
            break
        else:
            # Feed the REAL error message back to the LLM.
            # This is the key difference from self-reflection:
            # the feedback is ground truth, not LLM opinion.
            error_msg = result.stderr
            print(f"\n[External Test FAILED]\n{error_msg}")
            messages.append({"role": "assistant", "content": response.content})
            messages.append({
                "role": "user",
                "content": (
                    f"Your code failed the tests with this error:\n\n"
                    f"{error_msg}\n\n"
                    f"Fix the function. Return ONLY the corrected Python function."
                )
            })
    except subprocess.TimeoutExpired:
        print("\n[TIMEOUT] Code took too long — possible infinite loop.")
        messages.append({"role": "assistant", "content": response.content})
        messages.append({
            "role": "user",
            "content": "Your code timed out. It might have an infinite loop. Fix it."
        })
    finally:
        os.unlink(tmp_path)  # Clean up the temp file

print("\n--- External Reflection Complete ---")

Level 3: EXTERNAL REFLECTION (Test Execution)

────────────────────────────────────────
Attempt 1
────────────────────────────────────────

[Generated Code]
def is_palindrome(s: str) -> bool:
    cleaned = ''.join(ch.lower() for ch in s if ch.isalnum())
    return cleaned == cleaned[::-1]

[External Test Output]
ALL TESTS PASSED

✓ All tests passed — code verified by REAL execution!

--- External Reflection Complete ---


### 2.4 Comparison & Discussion

| Level | Feedback Source | Reliability | Best For |
|-------|---------------|-------------|----------|
| **Non-Reflection** | None | Low — blind trust | Quick drafts, low-stakes tasks |
| **Self-Reflection** | LLM itself | Medium — can miss its own bugs | Writing, design, subjective tasks |
| **External Reflection** | Real execution | **Highest** — ground truth | Code, math, anything with testable correctness |

**Key insight:** For tasks with **objective correctness** (code, math, data processing), external reflection is far more reliable. For **subjective tasks** (writing, design), self-reflection or two-agent reflection (Section 1's critique pattern) works well.

---
## Section 3: Planning — Construct a Plan and Execute It

### 3.1 Key Idea

```
Before:  Human designs steps → LLM executes each step
Now:     Human gives goal    → LLM designs AND executes steps
```

This is how models like Gemini Thinking and o1 work internally — they **plan before acting**.

Our approach has three phases:
1. **Phase 1 — Plan:** LLM generates a structured plan as JSON (with dependencies)
2. **Phase 2 — Execute:** LLM executes each step, passing results from completed steps as context
3. **Phase 3 — Synthesize:** LLM combines all step outputs into a final deliverable

### 3.2 Exercise: Brand Identity Creation

> **What to observe:** Watch how the LLM breaks down a vague creative task into concrete, ordered steps — and how each step builds on the previous ones.

In [ ]:
import json

TASK = (
    "Create a complete brand identity concept for a new bubble tea shop "
    "called 'Boba Bliss' in Hong Kong. Include: brand values, target audience, "
    "color palette, logo description, tagline, and a sample social media post."
)

# ══════════════════════════════════════════════════════════════
# PHASE 1: LLM creates its own plan
#
# We ask the LLM to output a JSON plan. Each step has:
# - step_number: order of execution
# - title: short name
# - description: what to do
# - depends_on: which steps must complete first
#
# The JSON format gives us structure we can programmatically
# iterate over. XML or even Python code would also work.
# ══════════════════════════════════════════════════════════════
plan_prompt = (
    f"You are a project manager. Given the following task, create a detailed "
    f"execution plan as a JSON array. Each step should have:\n"
    f"- 'step_number': integer\n"
    f"- 'title': short title\n"
    f"- 'description': what to do in this step\n"
    f"- 'depends_on': list of step numbers this depends on (empty list if none)\n\n"
    f"Task: {TASK}\n\n"
    f"Return ONLY the JSON array, no other text."
)

print("=" * 60)
print("PHASE 1: LLM Creates Its Own Plan")
print("=" * 60)

plan_response = chat(
    [{"role": "user", "content": plan_prompt}],
)

# Parse the JSON plan (handle markdown wrapping)
plan_text = plan_response.content
if "```json" in plan_text:
    plan_text = plan_text.split("```json")[1].split("```")[0]
elif "```" in plan_text:
    plan_text = plan_text.split("```")[1].split("```")[0]

try:
    plan = json.loads(plan_text)
except json.JSONDecodeError:
    print("Failed to parse plan. Raw output:")
    print(plan_response.content)
    plan = []

print("\n📋 Generated Plan:")
for step in plan:
    deps = step.get('depends_on', [])
    dep_str = f" (depends on step {deps})" if deps else ""
    print(f"  Step {step['step_number']}: {step['title']}{dep_str}")
    print(f"         {step['description']}")

PHASE 1: LLM Creates Its Own Plan

📋 Generated Plan:
  Step 1: Project Kick-off
         Hold a kickoff meeting with project stakeholders to confirm objectives, scope (complete brand identity for 'Boba Bliss' in Hong Kong), timeline, budget, decision-makers, and deliverables. Gather any existing research, inspiration, legal constraints (name checks, trademarks) and brand requirements.
  Step 2: Market & Competitive Research (depends on step [1])
         Research Hong Kong bubble tea market: size, growth, neighborhood hotspots (e.g., Mong Kok, Causeway Bay), local consumer behavior, peak times, pricing. Analyze 6-8 direct competitors and note brand positioning, design cues, menu, and promotion tactics. Summarize trends (seasonal flavors, sustainability, tea sourcing, Instagrammability).
  Step 3: Customer Research (depends on step [1])
         Collect primary data via short online survey and 8-12 interviews/field observations with Hong Kong consumers (students, young professionals, fa

In [ ]:
from openai import BadRequestError

# ══════════════════════════════════════════════════════════════
# PHASE 2: Execute each step, passing context from dependencies
#
# For each step, we:
# 1. Gather outputs from steps it depends on (the 'depends_on' field)
# 2. Build a prompt with the step description + dependency context
# 3. Call the LLM to execute that step
# 4. Store the result for downstream steps to use
#
# Azure's content filter can false-positive on large accumulated
# marketing/branding text, so we cap context size and retry on
# filter errors with a shorter prompt.
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("PHASE 2: Execute Each Step")
print("=" * 60)

MAX_CONTEXT_CHARS = 1500

results = {}  # step_number → output text
for step in plan:
    step_num = step["step_number"]
    print(f"\n{'─'*40}")
    print(f"▶ Executing Step {step_num}: {step['title']}")
    print(f"{'─'*40}")

    # Gather context from dependency steps (capped per-dep)
    context = ""
    if step.get("depends_on"):
        per_dep_limit = MAX_CONTEXT_CHARS // max(len(step["depends_on"]), 1)
        for dep in step["depends_on"]:
            if dep in results:
                snippet = results[dep][:per_dep_limit]
                context += f"\n[Result from Step {dep} (summary)]:\n{snippet}\n"

    exec_prompt = (
        f"You are executing Step {step_num} of a brand identity plan.\n"
        f"Step title: {step['title']}\n"
        f"Step description: {step['description']}\n"
        f"Overall task: {TASK}\n"
    )
    if context:
        exec_prompt += f"\nContext from previous steps:{context}"
    exec_prompt += "\nProvide a detailed, creative output for this step."

    try:
        step_result = chat(
            [{"role": "user", "content": exec_prompt}],
        )
    except BadRequestError:
        print("  ⚠ Content filter triggered — retrying with shorter context...")
        short_prompt = (
            f"You are executing Step {step_num} of a brand identity plan.\n"
            f"Step title: {step['title']}\n"
            f"Step description: {step['description']}\n"
            f"Overall task: {TASK}\n"
            f"\nProvide a detailed, creative output for this step."
        )
        try:
            step_result = chat(
                [{"role": "user", "content": short_prompt}],
            )
        except BadRequestError:
            print("  ✗ Content filter still triggered — skipping this step.")
            results[step_num] = "(Step skipped due to content filter.)"
            continue

    results[step_num] = step_result.content
    display_text = step_result.content[:400] + "..." if len(step_result.content) > 400 else step_result.content
    print(display_text)

PHASE 2: Execute Each Step

────────────────────────────────────────
▶ Executing Step 1: Project Kick-off
────────────────────────────────────────
Project Kickoff — Boba Bliss (Hong Kong)
Meeting date: 2026-04-15
Attendees: Founder (Ms. Chan), Ops Manager (Mr. Lee), Marketing Lead (Ms. Wong), Investor Rep (Mr. Lau), Creative Lead (you/agency), Legal Consultant (advisory)

Confirmed objectives
- Create a complete, deployable brand identity for a new bubble tea shop in Hong Kong called “Boba Bliss” (English) with a compact Traditional Chinese ...

────────────────────────────────────────
▶ Executing Step 2: Market & Competitive Research
────────────────────────────────────────
Below is Step 2: Market & Competitive Research plus a complete brand identity concept for Boba Bliss (Hong Kong). I’ve combined market context, competitor analysis, trends, and the brand outputs you requested so the creative direction ties directly to local opportunity.

1) Hong Kong bubble tea market — snapshot & 

In [ ]:
# ══════════════════════════════════════════════════════════════
# PHASE 3: Synthesize all results into a final deliverable
#
# We gather all step outputs and ask the LLM to combine them
# into one cohesive document. This is the "assembly" step.
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("PHASE 3: Final Synthesis")
print("=" * 60)

all_results = "\n\n".join([
    f"## Step {k}: {plan[k-1]['title']}\n{v}"
    for k, v in results.items()
])

synthesis = chat([{
    "role": "user",
    "content": (
        f"Combine all the following sections into a cohesive, "
        f"professional brand identity document. "
        f"Format it nicely with headers and sections:\n\n{all_results}"
    )
}])
print(synthesis.content)

PHASE 3: Final Synthesis
Boba Bliss — Brand Identity Pack
Project: Boba Bliss (Hong Kong) — integrated brand identity, launch assets, and handoff
Prepared: 2026-04-15
Prepared by: Creative Lead / Agency

OVERVIEW
This document brings together the full brand identity work for Boba Bliss: strategy, research insights, creative direction, visual system, rollout plan, deliverables, and handoff assets. It’s designed as a single reference for the founder, operations, marketing, investor, legal and creative teams to execute a smooth brand launch in Hong Kong.

CONTENTS (quick)
1. Executive summary
2. Project scope, timeline & budget
3. Legal & technical immediate actions
4. Market & competitive insights
5. Customer research & personas
6. Positioning, brand essence & values
7. Brand personality & tone-of-voice
8. Naming (English + Traditional Chinese) — options & recommendation
9. Tagline selection & usage
10. Visual identity — logo system, color palette, typography
11. Application examples — p

### 3.3 Discussion

1. **Autonomy:** The LLM created the plan **on its own**. We only provided the goal. This is the path toward highly autonomous agents.

2. **What if the plan is bad?** You could add a "plan review" step — ask a second LLM call to critique the plan before execution. This is **Planning + Reflection** combined.

3. **Parallel execution:** Steps with no dependencies (`depends_on: []`) could theoretically run in parallel using `asyncio.gather`. This would speed things up significantly.

4. **Plan formats:** We used JSON, but XML, YAML, or even Python code all work. LLMs are fine-tuned to be familiar with structured planning tasks, so they handle these formats well.

---
## Section 4: Multi-Agent Systems with AutoGen

### 4.1 Why Multiple Agents Instead of One?

Three key reasons:

| Problem with Single Agent | Multi-Agent Solution |
|--------------------------|---------------------|
| **Persona conflict:** "Be creative AND strictly follow safety rules" → model averages both, does neither well | **Split** into a creative Designer agent + a strict Safety Reviewer agent |
| **Context fog:** Performance drops sharply once context exceeds ~40% of the window ("Dumb Zone") | **Each agent** carries only its relevant context → stays in the "Smart Zone" |
| **Error cascade:** One mistake in Step 2 propagates through Steps 3–10 | **Agents as checkpoints** — a Critic agent catches errors before they cascade |

### 4.2 Two Communication Patterns

We'll implement both:

| Pattern | How It Works | Best For |
|---------|-------------|----------|
| **Manager (Hierarchical)** | A manager agent delegates tasks to worker agents | Well-defined tasks with clear responsibilities |
| **Group Chat** | All agents discuss in a shared conversation | Creative tasks needing debate and iteration |

### 4.3 Exercise A: Manager Pattern — Manager Calls Other Agents

> **What to observe:** Watch how the Manager creates a delegation plan, dispatches tasks to specialized workers, and synthesizes their outputs. Each worker only sees its own task — this keeps them in the "Smart Zone."

In [ ]:
# ══════════════════════════════════════════════════════════════
# MULTI-AGENT: Manager Pattern
#
# Architecture:
#   Manager Agent
#     ├── Content Writer (worker)
#     ├── SEO Expert (worker)
#     └── Social Media Manager (worker)
#
# Flow:
#   1. Manager receives the goal
#   2. Manager creates a delegation plan (who does what)
#   3. Manager dispatches tasks to workers one by one
#   4. Manager synthesizes all worker outputs
#
# This pattern mirrors real organizations.
# ══════════════════════════════════════════════════════════════

class SimpleAgent:
    """A lightweight agent wrapper: name + system prompt + run()."""
    def __init__(self, name: str, system_prompt: str):
        self.name = name
        self.system_prompt = system_prompt

    def run(self, task: str) -> str:
        """Execute a task using this agent's persona."""
        response = chat([
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": task}
        ])
        return response.content

# ── Define specialized worker agents ──────────────────────────
# Each agent has a narrow persona → better quality output
# (Remember: "Persona Focus" from slide 14)
writer = SimpleAgent(
    name="Content Writer",
    system_prompt=(
        "You are a professional content writer specializing in tech products. "
        "You write engaging, clear, and persuasive copy. "
        "Always match the tone to the target audience."
    )
)

seo_expert = SimpleAgent(
    name="SEO Expert",
    system_prompt=(
        "You are an SEO specialist. You optimize content for search engines. "
        "You suggest keywords, meta descriptions, heading structures, "
        "and internal linking strategies. Be specific with keyword suggestions."
    )
)

social_media_mgr = SimpleAgent(
    name="Social Media Manager",
    system_prompt=(
        "You are a social media manager. You create platform-specific content "
        "for Instagram, Twitter/X, LinkedIn, and TikTok. You know optimal "
        "post lengths, hashtag strategies, and engagement hooks for each platform."
    )
)

# Registry so the manager can look up workers by name
WORKERS = {
    "Content Writer": writer,
    "SEO Expert": seo_expert,
    "Social Media Manager": social_media_mgr,
}

print("Worker agents defined ✓")
for name in WORKERS:
    print(f"  • {name}")

Worker agents defined ✓
  • Content Writer
  • SEO Expert
  • Social Media Manager


In [ ]:
def run_manager(goal: str):
    """
    Manager agent: receives a goal, plans delegation,
    dispatches to workers, and synthesizes results.
    """
    print(f"\n{'='*60}")
    print(f"GOAL: {goal}")
    print(f"{'='*60}")

    # ── Step 1: Manager creates delegation plan ────────────────
    # The manager uses its knowledge of the team to decide who
    # does what and in what order.
    plan_prompt = (
        f"You are a marketing team manager. Your team members are:\n"
        f"1. Content Writer - writes articles, blog posts, product descriptions\n"
        f"2. SEO Expert - optimizes content for search engines\n"
        f"3. Social Media Manager - creates social media posts\n\n"
        f"Given the goal below, create a delegation plan as a JSON array.\n"
        f"Each item: {{'agent': 'exact agent name', 'task': 'specific task', 'order': 1}}\n\n"
        f"Goal: {goal}\n\nReturn ONLY the JSON array."
    )

    plan_response = chat([{"role": "user", "content": plan_prompt}])
    plan_text = plan_response.content
    if "```json" in plan_text:
        plan_text = plan_text.split("```json")[1].split("```")[0]
    elif "```" in plan_text:
        plan_text = plan_text.split("```")[1].split("```")[0]

    try:
        delegation_plan = json.loads(plan_text)
    except json.JSONDecodeError:
        print("Failed to parse delegation plan:")
        print(plan_response.content)
        return

    print("\n📋 [Manager's Delegation Plan]")
    for item in delegation_plan:
        print(f"  {item['order']}. {item['agent']} → {item['task']}")

    # ── Step 2: Dispatch tasks to workers ──────────────────────
    # Workers execute in order. Later workers receive context
    # from earlier workers so they can build on each other's work.
    all_results = {}
    sorted_plan = sorted(delegation_plan, key=lambda x: x["order"])

    for item in sorted_plan:
        agent_name = item["agent"]
        task = item["task"]

        print(f"\n{'─'*40}")
        print(f"📤 [Manager → {agent_name}]")
        print(f"   Task: {task}")
        print(f"{'─'*40}")

        agent = WORKERS.get(agent_name)
        if agent:
            # Pass context from earlier workers
            context = ""
            if all_results:
                context = "\n\nContext from other team members:\n"
                for name, r in all_results.items():
                    context += f"[{name}]: {r[:200]}...\n"
            result = agent.run(task + context)
            all_results[agent_name] = result
            print(result[:400] + "..." if len(result) > 400 else result)
        else:
            print(f"  ⚠ Unknown agent: {agent_name}")

    # ── Step 3: Manager synthesizes all outputs ────────────────
    print(f"\n{'='*60}")
    print("📊 [Manager: Final Synthesis]")
    print(f"{'='*60}")

    synthesis_prompt = (
        "You are a marketing manager. Your team has completed their tasks.\n"
        "Synthesize their work into a final, cohesive marketing deliverable.\n\n"
    )
    for name, r in all_results.items():
        synthesis_prompt += f"[{name}]:\n{r}\n\n"

    final = chat([{"role": "user", "content": synthesis_prompt}])
    print(final.content)


# ── Run the Manager Pattern! ──────────────────────────────────
run_manager(
    "Launch a marketing campaign for a new AI-powered design tool "
    "called 'DesignMate' that helps university students create "
    "professional presentations in minutes."
)


GOAL: Launch a marketing campaign for a new AI-powered design tool called 'DesignMate' that helps university students create professional presentations in minutes.

📋 [Manager's Delegation Plan]
  1. Content Writer → Develop campaign messaging and student personas: define 3 core value propositions, tone of voice, 5 university student personas, elevator pitch and key messages for landing page, blog, and social.
  2. SEO Expert → Perform keyword research targeting university students and presentation needs: deliver prioritized list of head and long‑tail keywords, search intent notes, monthly volume estimates, competitor keyword gaps, and recommended primary keywords for landing page and blog topics.
  3. Content Writer → Write landing page copy: headline, subhead, benefits, feature descriptions, CTAs, 3 student-focused product descriptions, and an FAQ optimized for student concerns.
  4. SEO Expert → Optimize landing page for search and UX: implement title tags, meta descriptions, schem

### 4.4 Exercise B: Group Chat — Agents Discuss and Collaborate (AutoGen)

Three agents collaborate on designing a mobile app UI:
- **Designer** — proposes visual designs (creative role)
- **UX Researcher** — critiques for usability & accessibility (critical role)
- **Project Manager** — resolves conflicts and approves the final design (decision-maker)

They take turns in a **round-robin** group chat. The conversation ends when the PM says "APPROVED" or after 12 messages.

> **What to observe:** Watch how the three agents naturally debate — the Designer proposes, the UX Researcher pushes back, and the PM mediates. This "creative tension" produces better results than any single agent could.

In [10]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_ext.models.openai import AzureOpenAIChatCompletionClient

# ── AutoGen Model Client ──────────────────────────────────────
# AutoGen needs its own model client wrapper.
# We use AzureOpenAIChatCompletionClient to connect to HKUST's API.
model_client = AzureOpenAIChatCompletionClient(
    azure_deployment=MODEL_NAME,   # The deployment name on Azure
    model=MODEL_NAME,              # The model name for AutoGen's internal use
    api_key=AZURE_API_KEY,
    azure_endpoint=AZURE_ENDPOINT,
    api_version=AZURE_API_VERSION,
)

# ── Define Agents ─────────────────────────────────────────────
# Each agent has a distinct persona via its system_message.
# AutoGen uses these system messages to shape each agent's behavior.

designer = AssistantAgent(
    name="Designer",
    model_client=model_client,
    system_message=(
        "You are a senior UI/UX designer. You propose visual designs, "
        "color schemes, layouts, and interaction patterns. You care deeply "
        "about aesthetics, consistency, and modern design trends. "
        "Be specific about colors (hex codes), typography, and spacing."
    ),
)

ux_researcher = AssistantAgent(
    name="UX_Researcher",
    model_client=model_client,
    system_message=(
        "You are a UX researcher with expertise in user behavior and "
        "accessibility. You critique designs based on usability heuristics "
        "(Nielsen's 10), accessibility (WCAG 2.1), and user psychology. "
        "You always ask: 'Will the user understand this?' and "
        "'Does this work for users with disabilities?' "
        "Be specific and cite usability principles."
    ),
)

project_manager = AssistantAgent(
    name="Project_Manager",
    model_client=model_client,
    system_message=(
        "You are a project manager who ensures the team stays on track. "
        "You synthesize the Designer's proposals and the UX Researcher's "
        "feedback into actionable decisions. You resolve conflicts between "
        "aesthetics and usability. When the team has reached a good solution, "
        "summarize the final decision and say 'APPROVED' to end the discussion."
    ),
)

# ── Termination Conditions ────────────────────────────────────
# The chat stops when EITHER condition is met:
#   1. Someone says "APPROVED" (the PM's signal that we're done)
#   2. We reach 12 messages (safety limit to prevent runaway chats)
termination = TextMentionTermination("APPROVED") | MaxMessageTermination(12)

# ── Create the Group Chat ─────────────────────────────────────
# RoundRobinGroupChat: agents take turns in order
# (Designer → UX_Researcher → Project_Manager → Designer → ...)
team = RoundRobinGroupChat(
    participants=[designer, ux_researcher, project_manager],
    termination_condition=termination,
)

print("AutoGen Group Chat Team created ✓")
print("  Agents: Designer → UX_Researcher → Project_Manager")
print("  Termination: 'APPROVED' keyword OR 12 messages max")

AutoGen Group Chat Team created ✓
  Agents: Designer → UX_Researcher → Project_Manager
  Termination: 'APPROVED' keyword OR 12 messages max


In [11]:
# ── Run the Group Chat ────────────────────────────────────────
task = (
    "Design the home screen for a new fitness tracking app called 'FitFlow'. "
    "The app targets university students aged 18-25. Key features: "
    "daily step counter, workout logging, social challenges with friends, "
    "and a motivational quote of the day. The design should feel modern, "
    "energetic, and not overwhelming."
)

print("=" * 60)
print("MULTI-AGENT GROUP CHAT: FitFlow App Design")
print("=" * 60)
print(f"\nTask: {task}\n")

# In Google Colab, 'await' works directly at the top level
# because Colab's IPython kernel already runs an event loop.
result = await team.run(task=task)

# ── Print the full conversation log ───────────────────────────
print("\n" + "=" * 60)
print("FULL CONVERSATION LOG")
print("=" * 60)
for msg in result.messages:
    source = msg.source if hasattr(msg, 'source') else 'unknown'
    content = msg.content if isinstance(msg.content, str) else str(msg.content)
    print(f"\n🗣️ [{source}]")
    print(content[:600] + "..." if len(content) > 600 else content)
    print("-" * 40)

MULTI-AGENT GROUP CHAT: FitFlow App Design

Task: Design the home screen for a new fitness tracking app called 'FitFlow'. The app targets university students aged 18-25. Key features: daily step counter, workout logging, social challenges with friends, and a motivational quote of the day. The design should feel modern, energetic, and not overwhelming.



/usr/local/lib/python3.12/dist-packages/autogen_agentchat/agents/_assistant_agent.py:1109: UserWarning: Resolved model mismatch: gpt-5-mini-2025-08-07 != gpt-5-mini. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-5-mini to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(



FULL CONVERSATION LOG

🗣️ [user]
Design the home screen for a new fitness tracking app called 'FitFlow'. The app targets university students aged 18-25. Key features: daily step counter, workout logging, social challenges with friends, and a motivational quote of the day. The design should feel modern, energetic, and not overwhelming.
----------------------------------------

🗣️ [Designer]
Summary
- Goal: a modern, energetic, but calm home screen for FitFlow aimed at 18–25 year olds. Bright but not loud; friendly geometry and soft rounded shapes; clear hierarchy so students can glance and act quickly.
- Key pieces on home: top greeting + avatar, daily step ring + steps count, quick workout logging actions, social challenges carousel, friends streaks strip, and a small “Motivational Quote of the Day” card.

Color palette (tokens)
- Background: Soft neutral — #F6F9FF
- Surface / cards: White — #FFFFFF
- Primary gradient (used for step ring, primary CTAs): Coral → orange
  - start:...
--

### 4.5 Comparing Manager Pattern vs. Group Chat

| Aspect | Manager Pattern (4.3) | Group Chat (4.4) |
|--------|----------------------|------------------|
| **Control** | Centralized — manager decides everything | Decentralized — agents take turns |
| **Communication** | Hub-and-spoke (manager ↔ each worker) | All-to-all (everyone sees everything) |
| **Context per agent** | Workers only see their own task (lean) | All agents see full conversation (rich but heavier) |
| **Best for** | Well-defined tasks with clear delegation | Creative tasks needing debate & iteration |
| **Risk** | Manager becomes a bottleneck / single point of failure | Conversations can go off-track or loop |
| **Implementation** | Simple to build from scratch | Use frameworks like AutoGen |

---
## Bonus Challenge (Take-Home)

**Build a "Research Paper Summarizer" multi-agent system** with AutoGen:

1. **Paper Reader Agent** — Takes an abstract and extracts key claims
2. **Methodology Critic Agent** — Reviews the methodology for weaknesses
3. **Plain Language Translator Agent** — Rewrites the summary for a non-expert audience
4. **Manager Agent** — Coordinates the workflow and produces a final report

Combine the patterns from today:
- **Planning** for structuring the workflow
- **Reflection** for quality checking
- **Multi-agent** for collaboration

In [ ]:
# YOUR BONUS CODE HERE
# Build the Research Paper Summarizer multi-agent system
# Hint: You can use either the Manager pattern or AutoGen Group Chat
# ...